In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.api.types import CategoricalDtype

musinsa_lululemon_reviews = pd.read_csv('./data/musinsa_lululemon_reviews.csv')

In [2]:
musinsa_lululemon_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1944 entries, 0 to 1943
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   collect_date    1944 non-null   str    
 1   review_date     1944 non-null   str    
 2   product_id      1944 non-null   int64  
 3   product_name    1944 non-null   str    
 4   review_no       1944 non-null   int64  
 5   review_type     1944 non-null   str    
 6   rating          1944 non-null   int64  
 7   review_text     1944 non-null   str    
 8   author          1944 non-null   str    
 9   user_height     1600 non-null   float64
 10  user_weight     1600 non-null   float64
 11  option          1944 non-null   str    
 12  helpful         1944 non-null   int64  
 13  comment_count   1944 non-null   int64  
 14  has_image       1944 non-null   bool   
 15  image_urls      1181 non-null   str    
 16  survey_size     0 non-null      float64
 17  survey_width    0 non-null      float64
 18 

In [3]:
musinsa_lululemon_reviews.shape

(1944, 21)

### 컬럼명 변경

In [4]:
musinsa_lululemon_reviews = musinsa_lululemon_reviews.rename(columns={
    'review_text' : 'content',
    'helpful' : 'helpful_count',
    'option' : 'purchase_option'
})

### ```year```, ```month``` 추출

In [5]:
musinsa_lululemon_reviews = musinsa_lululemon_reviews.copy()

# 날짜형으로 변환
musinsa_lululemon_reviews['review_date'] = pd.to_datetime(musinsa_lululemon_reviews['review_date'], errors='coerce')

# 연도 / 월 컬럼 생성
musinsa_lululemon_reviews['year'] = musinsa_lululemon_reviews['review_date'].dt.year
musinsa_lululemon_reviews['month'] = musinsa_lululemon_reviews['review_date'].dt.month

In [6]:
musinsa_lululemon_reviews

,collect_date,review_date,product_id,product_name,review_no,review_type,rating,content,author,user_height,...,comment_count,has_image,image_urls,survey_size,survey_width,survey_comfort,survey_instep,url,year,month
0,2026-04-21 22:57:48,2026-04-09,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83547450,일반,4,룰루레몬 헤어밴드 좋습니다. 러닝할때 아주 잘 사용해요,산만한브라운정장,NaN,...,0,True,https://www.musinsa.com/data/estimate/5733085_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...,2026,4
1,2026-04-21 22:57:48,2026-04-04,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83449647,일반,5,땀이 많은 편이라 러닝 한번 하고나면 상의가 반쯤 젖을만큼 땀을 흘리는데 확실히 밴...,None27,NaN,...,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...,2026,4
2,2026-04-21 22:57:48,2026-04-04,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83439220,일반,5,안벗겨지고 깔끔하고 무난합니다. 땀도 잘 흡수하고 가볍고 빨리 마르고 착용감도 너어...,비범한제노바티켓,NaN,...,0,True,https://www.musinsa.com/data/estimate/5733085_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...,2026,4
3,2026-04-21 22:57:48,2026-04-04,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83189716,일반,5,벌써 여러 가지 색상으로 룰루레몬에 헤어밴드를 많이 가지고 있습니다. 매우 만족하며...,헬미온,NaN,...,0,True,https://www.musinsa.com/data/estimate/5733085_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...,2026,4
4,2026-04-21 22:57:48,2026-04-01,5733085,심리스 와이드 헤드밴드 SOGY VPOR,83375899,일반,5,운동할 때 쓰려고 샀는데 진짜 너무 좋아요 재질도 너무 만족스럽네요,뭔상관ㅋ,NaN,...,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5733085#revie...,2026,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1939,2026-04-21 23:38:54,2026-04-20,5732264,댄스 스튜디오 미드라이즈 조거 *아시아 핏 BLK,83817354,일반,5,배송도 빠르고 상품도 너무 좋아요 재구매 의사가 있어요,박정상입니다,170.0,...,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5732264#revie...,2026,4
1940,2026-04-21 23:38:58,2026-03-16,5732245,올 잇 테익스 리브드 Nulu 롱슬리브 셔츠,82995590,일반,5,최근에 룰루레몬 제품 너무 홀릭되어 여러벌 구매했습니다. 운동할 때 진짜 편해서 자...,똑똑한동대문힙본,163.0,...,0,True,https://www.musinsa.com/data/estimate/5732245_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5732245#revie...,2026,3
1941,2026-04-21 23:39:03,2026-04-08,5732221,소프트 저지 하프집 HTTN HCLU,83536790,일반,5,"미국에서 다른색 구매하고 너무 부드럽고 만족해서, 다른 색상 추가로 구매합니다.",wonbaek,181.0,...,0,True,https://www.musinsa.com/data/estimate/5732221_...,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5732221#revie...,2026,4
1942,2026-04-21 23:39:08,2026-04-04,5731952,데이드리프트 하이라이즈 트라우저 *레귤러 WALC,83437946,일반,5,운동복의 편안함과 슬랙스의 단정함을 동시에 잡고 싶은 날 손이 가장 많이 가는 데이...,shoppig,173.0,...,0,False,NaN,NaN,NaN,NaN,NaN,https://www.musinsa.com/products/5731952#revie...,2026,4


### 없는 컬럼 추가

In [7]:
# 없는 컬럼 추가하기
required_columns = [
    'collect_date', 'platform', 'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'content', 'rating', 'helpful_count',
    'has_image', 'purchase_option', 'user_height', 'user_weight',
    'user_height_group', 'user_weight_group', 'product_url'
]

for col in required_columns:
    if col not in musinsa_lululemon_reviews.columns:
        musinsa_lululemon_reviews[col] = None

# platform 값 추가
musinsa_lululemon_reviews['platform'] = '무신사'

# 컬럼 순서 정리
musinsa_lululemon_reviews = musinsa_lululemon_reviews[required_columns]

### 키 값 이름 변경
- 190cm -> 190cm 이상
- 150cm -> 150cm 이하 변경

In [8]:
musinsa_lululemon_reviews['user_height_group'] = musinsa_lululemon_reviews['user_height_group'].replace({
    '190cm' : '190cm 이상',
    '150cm' : '150cm 이하'
})

### 몸무게 값 이름 변경
- 90kg -> 90kg 이상
- 44kg -> 44kg 이하 변경

In [9]:
musinsa_lululemon_reviews['user_height_group'].value_counts()

Series([], Name: count, dtype: int64)

In [10]:
musinsa_lululemon_reviews['user_weight_group'] = musinsa_lululemon_reviews['user_weight_group'].replace({
    '90kg' : '90kg 이상',
    '44kg' : '44kg 이하'
})

### 결측치 % 확인

In [11]:
(musinsa_lululemon_reviews.isna().mean() * 100).sort_values(ascending=False)

product_url          100.000000
review_id            100.000000
user_weight_group    100.000000
user_height_group    100.000000
user_weight           17.695473
user_height           17.695473
helpful_count          0.000000
purchase_option        0.000000
has_image              0.000000
collect_date           0.000000
platform               0.000000
content                0.000000
month                  0.000000
year                   0.000000
review_date            0.000000
product_name           0.000000
product_id             0.000000
rating                 0.000000
dtype: float64

### ```review_id``` 중복 확인

In [12]:
# 중복 확인
print(f"전체 행 수: {len(musinsa_lululemon_reviews):,}")
print(f"review_id 기준 중복 수: {musinsa_lululemon_reviews.duplicated(subset=['review_id']).sum():,}")
print(f"review_id 고유값 수: {musinsa_lululemon_reviews['review_id'].nunique():,}")

전체 행 수: 1,944
review_id 기준 중복 수: 1,943
review_id 고유값 수: 0


### 중복 리뷰 데이터 확인 및 제거

In [13]:
multi_name = musinsa_lululemon_reviews.groupby('product_id')['product_name'].nunique()
multi_url = musinsa_lululemon_reviews.groupby('product_id')['product_url'].nunique()

# product_name 매핑 확인
print(multi_name.gt(1).sum(), "개의 product_id가 여러 product_name에 매핑됨")
display(
    musinsa_lululemon_reviews[
        musinsa_lululemon_reviews['product_id'].isin(multi_name[multi_name > 1].index)
    ][['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

# product_url 매핑 확인
print(multi_url.gt(1).sum(), "개의 product_id가 여러 product_url에 매핑됨")
display(
    musinsa_lululemon_reviews[
        musinsa_lululemon_reviews['product_id'].isin(multi_url[multi_url > 1].index)
    ][['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

0 개의 product_id가 여러 product_name에 매핑됨


,product_id,product_name,product_url


0 개의 product_id가 여러 product_url에 매핑됨


,product_id,product_name,product_url


In [14]:
# ===============================
# 제품명 매칭 확인
# ===============================
multi_name = musinsa_lululemon_reviews.groupby('product_id')['product_name'].nunique()
multi_url = musinsa_lululemon_reviews.groupby('product_id')['product_url'].nunique()

# product_name 매핑 확인
print(multi_name.gt(1).sum(), "개의 product_id가 여러 product_name에 매핑됨")
display(
    musinsa_lululemon_reviews[musinsa_lululemon_reviews['product_id'].isin(multi_name[multi_name > 1].index)]
    [['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

# product_url 매핑 확인
print(multi_url.gt(1).sum(), "개의 product_id가 여러 product_url에 매핑됨")
display(
    musinsa_lululemon_reviews[musinsa_lululemon_reviews['product_id'].isin(multi_url[multi_url > 1].index)]
    [['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

0 개의 product_id가 여러 product_name에 매핑됨


,product_id,product_name,product_url


0 개의 product_id가 여러 product_url에 매핑됨


,product_id,product_name,product_url


In [15]:
# ===============================
# 리뷰 내용 중복 확인
# ===============================
# 2-1. product_id 같고 content 중복인 데이터 확인
dup_mask = musinsa_lululemon_reviews.duplicated(subset=['product_id', 'content'], keep=False)
print("product_id는 같고 content가 중복 개수: ", dup_mask.sum())
display(musinsa_lululemon_reviews[dup_mask][['product_id', 'product_name', 'review_date', 'content']].sort_values(['product_id', 'content']).head(20))

# 2-2. 위 경우 중 review_date까지 완전히 동일한 경우 개수
print(
    "review_date까지 완전히 동일한 경우 개수: "
    , musinsa_lululemon_reviews[dup_mask].duplicated(subset=['product_id', 'content', 'review_date'], keep=False).sum()
)

# 2-3. product_id, content, review_date, purchase_option 까지 모두 중복인 경우
dup_mask_full = musinsa_lululemon_reviews.duplicated(
    subset=['product_id', 'content', 'review_date', 'purchase_option'],
    keep=False
)
print("purchase_option까지 완전히 동일한 경우 개수: ", dup_mask_full.sum())
display(
    musinsa_lululemon_reviews[dup_mask_full][['product_id', 'product_name', 'review_date', 'purchase_option', 'content']]
    .sort_values(['product_id', 'review_date'])
    .head(20)
)

product_id는 같고 content가 중복 개수:  601


,product_id,product_name,review_date,content
475,5731786,더 매트 5mm *천연고무 사용 BLK,2026-03-04,가격만 좀더 저렴하면 너무 좋을것 같아요. 룰루레몬 매트 최고에요
1572,5731786,더 매트 5mm *천연고무 사용 BLK,2026-03-04,가격만 좀더 저렴하면 너무 좋을것 같아요. 룰루레몬 매트 최고에요
476,5731786,더 매트 5mm *천연고무 사용 BLK,2026-02-16,요가매트 추천 영상들 찾아보다가 룰루레몬거 구매했습니다 검은색 매트는 잘 없던데 검...
1573,5731786,더 매트 5mm *천연고무 사용 BLK,2026-02-16,요가매트 추천 영상들 찾아보다가 룰루레몬거 구매했습니다 검은색 매트는 잘 없던데 검...
189,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요
190,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요
510,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...
517,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...
477,5732234,더 매트 5mm *천연고무 사용 BLK WHT BLK,2026-01-23,무신사 오픈기념으로 구매하였습니다 룰루레몬을 취급하는 타 사이트에서는 할인 자체가 ...
1574,5732234,더 매트 5mm *천연고무 사용 BLK WHT BLK,2026-01-23,무신사 오픈기념으로 구매하였습니다 룰루레몬을 취급하는 타 사이트에서는 할인 자체가 ...


review_date까지 완전히 동일한 경우 개수:  585
purchase_option까지 완전히 동일한 경우 개수:  581


,product_id,product_name,review_date,purchase_option,content
476,5731786,더 매트 5mm *천연고무 사용 BLK,2026-02-16,Black · One Size,요가매트 추천 영상들 찾아보다가 룰루레몬거 구매했습니다 검은색 매트는 잘 없던데 검...
1573,5731786,더 매트 5mm *천연고무 사용 BLK,2026-02-16,Black · One Size,요가매트 추천 영상들 찾아보다가 룰루레몬거 구매했습니다 검은색 매트는 잘 없던데 검...
475,5731786,더 매트 5mm *천연고무 사용 BLK,2026-03-04,Black · One Size,가격만 좀더 저렴하면 너무 좋을것 같아요. 룰루레몬 매트 최고에요
1572,5731786,더 매트 5mm *천연고무 사용 BLK,2026-03-04,Black · One Size,가격만 좀더 저렴하면 너무 좋을것 같아요. 룰루레몬 매트 최고에요
189,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,Black · M,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요
190,5732069,소프트 저지 클래식 핏 팬츠 *레귤러 BLK,2026-04-09,Black · M,배송이 매우 빠르고 바지도 마음에 드네요 잘 입을게요
510,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,White · L,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...
517,5732140,메탈 벤트 테크 숏슬리브 셔츠 WHWH,2026-01-31,White · L,소재나 착용감은 정말 좋구요. 다만 사이즈가 저한테 조금 애매 하네요. 좀 더 여유...
477,5732234,더 매트 5mm *천연고무 사용 BLK WHT BLK,2026-01-23,Black/White/Black · One Size,무신사 오픈기념으로 구매하였습니다 룰루레몬을 취급하는 타 사이트에서는 할인 자체가 ...
1574,5732234,더 매트 5mm *천연고무 사용 BLK WHT BLK,2026-01-23,Black/White/Black · One Size,무신사 오픈기념으로 구매하였습니다 룰루레몬을 취급하는 타 사이트에서는 할인 자체가 ...


In [16]:
# ===============================
# 중복리뷰 제거
# ===============================
before = len(musinsa_lululemon_reviews)

musinsa_lululemon_reviews = (
    musinsa_lululemon_reviews
    .sort_values(
        by=['helpful_count', 'has_image', 'user_height', 'user_weight', 'user_height_group', 'user_weight_group'],
        ascending=[False, False, False, False, False, False],
        na_position='last'
    )
    .drop_duplicates(
        subset=['product_id', 'content', 'review_date', 'purchase_option'],
        keep='first'
    )
    .sort_index()
)

after = len(musinsa_lululemon_reviews)

print(f"제거 전: {before}개")
print(f"제거된 데이터: {before - after}개")
print(f"남은 데이터: {after}개")

제거 전: 1944개
제거된 데이터: 415개
남은 데이터: 1529개


### ```content``` 텍스트 정제

In [17]:
# HTML 태그 제거
HTML_PATTERN = re.compile(r'<[^>]+>')
# URL 제거
URL_PATTERN = re.compile(r'http\S+|www\.\S+')
# 이메일 제거
EMAIL_PATTERN = re.compile(r'\S+@\S+')
# 특수문자 제거
SPECIAL_PATTERN = re.compile(r'[^\w\s\uAC00-\uD7A3\u3130-\u318F!?]')
# 공백(줄바꿈, 탭) 정리
WHITESPACE_PATTERN = re.compile(r'\s+')
# 반복 문자 축소
REPEAT_PATTERN = re.compile(r'(.)\1{2,}')

def clean_text(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text)

    text = HTML_PATTERN.sub(' ', text)
    text = URL_PATTERN.sub(' ', text)
    text = EMAIL_PATTERN.sub(' ', text)
    text = SPECIAL_PATTERN.sub(' ', text)
    text = REPEAT_PATTERN.sub(r'\1\1', text)
    text = WHITESPACE_PATTERN.sub(' ', text)
# 앞뒤 공백 제거, 소문자화
    text = text.strip().lower()

    return text

musinsa_lululemon_reviews['content'] = musinsa_lululemon_reviews['content'].apply(clean_text)

### ```content``` 5글자 이하 제거

In [18]:
# =========================================
# 띄어쓰기 제거 후 5글자 이하인 데이터 확인
# =========================================
short_mask = musinsa_lululemon_reviews['content'].str.replace(' ', '', regex=False).str.len() <= 5

print(f"5글자 이하 행 수: {short_mask.sum()}")
display(musinsa_lululemon_reviews[short_mask]['content'].value_counts())

# =========================
# 5글자 이하 행 제거
# =========================
before = len(musinsa_lululemon_reviews)
musinsa_lululemon_reviews = musinsa_lululemon_reviews[~short_mask]
after = len(musinsa_lululemon_reviews)

print(f"제거 전: {before}개")
print(f"제거된 행: {before - after}개")
print(f"제거 후: {after}개")

5글자 이하 행 수: 0


Series([], Name: count, dtype: int64)

제거 전: 1529개
제거된 행: 0개
제거 후: 1529개


### ```purchase_option``` 텍스트 정제

In [19]:
musinsa_lululemon_reviews['purchase_option'].value_counts()

purchase_option
Black · M                  135
Black · L                   90
Black · S                   62
Black · One Size            56
Black · 4                   52
                          ... 
Obsidian · 32                1
Sheer Oak · XXS              1
True Navy/Club Blue · L      1
Walnut Crunch · XXXS         1
Graphite Grey · 30           1
Name: count, Length: 372, dtype: int64

In [20]:
# 정규식(패턴) 미리 컴파일 (반복 apply에서 성능 개선)
COLON_PATTERN = re.compile(r':\s+')
SLASH_PATTERN = re.compile(r'\s*/\s*')

def normalize_purchase_option(text):
    if pd.isna(text):
        return np.nan
    # 소문자 통일
    text = str(text).lower()

    # 콜론 뒤 공백 제거 (color: black → color:black)
    text = COLON_PATTERN.sub(':', text)

    # 슬래시 공백 통일 (a/b → a / b)
    text = SLASH_PATTERN.sub(' / ', text)

    # 앞뒤 공백 제거
    text = text.strip()

    return text


musinsa_lululemon_reviews['purchase_option'] = (
    musinsa_lululemon_reviews['purchase_option']
    .apply(normalize_purchase_option)
)

#### 사이즈 자동 정규화

### 파생컬럼 생성

#### ```women_size_yn```

In [21]:
musinsa_lululemon_reviews['women_size_yn'] = (
    musinsa_lululemon_reviews['purchase_option']
    .astype(str)
    .str.contains(r'\bw\b', case=False, na=False)
    .astype(int)
)

#### ```purchase_option_size```

#### ```purchase_option_color```

In [22]:
musinsa_lululemon_reviews['purchase_option']

0       solar grey / vapor · one size
1       solar grey / vapor · one size
2       solar grey / vapor · one size
3       solar grey / vapor · one size
4       solar grey / vapor · one size
                    ...              
1939                        black · l
1940                  light ivory · 6
1941        true navy / club blue · l
1942             walnut crunch · xxxs
1943               graphite grey · 30
Name: purchase_option, Length: 1529, dtype: str

In [33]:
musinsa_lululemon_reviews['purchase_option_color'].value_counts()

purchase_option_color
블랙                  571
화이트                  73
라이트_아이보리             65
그래파이트_그레이            43
솔라_그레이,베이퍼           39
                   ... 
라벤더_페인트,그레이프_미스트      1
다크_레드                 1
블랙,라이트_아이보리           1
클럽_블루,화이트             1
쉬어_오크                 1
Name: count, Length: 120, dtype: int64

In [32]:
musinsa_lululemon_reviews = musinsa_lululemon_reviews.copy()
col = musinsa_lululemon_reviews['purchase_option'].astype(str).str.lower()

# ---------------------------
# 컬러 번역
# ---------------------------
color_map = {
    'incline texture grey' : '인클라인 텍스처 그레이', # 트윌 그레이
    'serene blue' : '서린 블루',
    'aurora haze mini purple' : '오로라 헤이즈 미니 퍼플',
    'obsidian' : '옵시디언',
    'sequoia' : '세쿼이아',
    'vitapink' : '비타핑크',
    'pelican' : '펠리컨',
    'coconut ivory' : '코코넛 아이보리',
    'blue twill' : '블루 트윌',
    'downtown tan' : '다운타운 탄',
    'classic navy' : '클래식 네이비',
    'starch blue' : '스타치 블루',
    'black plum' : '블랙 플럼',
    'blush quartz' : '블러시 쿼츠',
    'meadow haze pink' : '메도우 헤이즈 핑크',
    'deep sea blue' : '딥 씨 블루',
    'clubhouse blue' : '클럽하우스 블루',
    'rainforest green' : '레인포레스트 그린',
    'gold' : '골드',
    'ivy grove' : '아이비 그로브',
    'breakfast tea' : '브렉퍼스트 티',
    'nomad' : '노마드',
    'true leopard nomad' : '트루 레오파드 노마드',
    'oxford red' : '옥스퍼드 레드',
    'polo pastel' : '폴로 파스텔',
    'foam cloud' : '폼 클라우드', #밀키 화이트
    'walnut crunch' : '월넛 크런치',
    'feather grey' : '페더 그레이',
    'fog green' : '포그 그린',
    'dove grey' : '도브 그레이',
    'asphalt grey' : '아스팔트 그레이',
    'willow leaf' : '윌로우 리프',
    'meadow haze' : '메도우 헤이즈',
    'prep blue' : '프렙 블루',
    'silver drop' : '실버 드롭',
    'speckled heather black' : '네프 블랙',
    'raw linen' : '내추럴 리넨', #오트밀
    'bone' : '본', #상아색
    'burgundy bay' : '버건디 베이',
    'blissful pink' : '블리스풀 핑크',
    'porcelain pink' : '포슬린 핑크',
    'pink pearl' : '핑크 펄',
    'lotus lavender' : '로터스 라벤더',
    'espresso' : '에스프레소',
    'onyx grey' : '오닉스 그레이',
    'true navy' : '트루 네이비', # 진네이비
    'brilliant blue' : '브릴리언트 블루',
    'sinatra blue' : '시나트라 블루',
    'core medium grey' : '코어 미디엄 그레이', # 중회색
    'indochine blue' : '인도차이나 블루',
    'rose gold' : '로즈 골드',
    'tea rose' : '티 로즈',
    'grey sage' : '세이지 그레이',
    'anchor' : '앵커',
    'warm ash grey' : '웜 애쉬 그레이',
    'faint lavender' : '라벤더 페인트',
    'club blue' : '클럽 블루',
    'sheer oak' : '쉬어 오크',
    'steel blue' : '스틸 블루',
    'core ultra light grey' : '코어 울트라 라이트 그레이',
    'ashen rose' : '애쉬 로즈',
    'slate' : '슬레이트', # 슬레이트 그레이
    'grape mist' : '그레이프 미스트',
    'light ivory' : '라이트 아이보리',
    'graphite grey' : '그래파이트 그레이',
    'solar grey' : '솔라 그레이',
    'vapor': '베이퍼',
    'dark navy': '다크_네이비',
    'dark blue': '다크_블루',
    'sky blue': '스카이_블루',
    'light blue': '라이트_블루',
    'off white': '오프_화이트',
    'light pink': '라이트_핑크',
    'dark pink': '다크_핑크',
    'dark red': '다크_레드',
    'dark gray': '다크_그레이',
    'dark grey': '다크_그레이',
    'melange gray': '멜란지_그레이',
    'sand white': '샌드_화이트',
    'light beige': '라이트_베이지',
    'oatmeal melange': '오트밀_멜란지',
    'black': '블랙',
    'brown': '브라운',
    'beige': '베이지',
    'white': '화이트',
    'cream': '크림',
    'blue': '블루',
    'navy': '네이비',
    'pink': '핑크',
    'red': '레드',
    'wine': '와인',
    'green': '그린',
    'khaki': '카키',
    'gray': '그레이',
    'grey': '그레이',
    'ivory': '아이보리',
    'purple': '퍼플',
    'yellow': '옐로우',
    'silver': '실버',
    'camel': '카멜',
    'orange': '오렌지'
}

# ---------------------------
# 사이즈 매핑
# ---------------------------
size_mapping = {
    'xxxs' : 85,
    'xxs': 90,
    'xs': 95,
    's': 95,
    'm': 100,
    'l': 105,
    'xl': 110,
    'xxl': 115,
    'xxxl': 120
}

# ---------------------------
# 컬러 추출
# ---------------------------
def extract_color(x):
    x = str(x).lower()

    # 구분자 통일
    x = re.sub(r'[·/]', ',', x)

    parts = [p.strip() for p in x.split(',') if p.strip()]

    result = []

    for p in parts:
        # 사이즈 제거 (단어 기준)
        p = re.sub(r'\b(one size|free|xxxs|xxs|xs|s|m|l|xl|xxl|xxxl)\b', '', p).strip()

        # 숫자 제거
        p = re.sub(r'\b\d+\b', '', p).strip()

        if not p:
            continue

        p = color_map.get(p, p)
        p = p.replace(' ', '_')

        result.append(p)

    return ','.join(result) if result else np.nan


# ---------------------------
# 사이즈 추출
# ---------------------------
def extract_size(x):
    x = str(x).lower()

    # one size / free
    if re.search(r'\b(one size|free)\b', x):
        return 0

    # 알파 사이즈
    for key, val in size_mapping.items():
        if re.search(r'\b' + key + r'\b', x):
            return val

    # 숫자 사이즈 (2~3자리)
    match = re.search(r'\b\d{2,3}\b', x)
    if match:
        return int(match.group())

    # 1자리 숫자 (예: 블랙,4)
    match = re.search(r'\b\d\b', x)
    if match:
        return int(match.group())

    return np.nan


# ---------------------------
# 적용
# ---------------------------
musinsa_lululemon_reviews['purchase_option_color'] = col.apply(extract_color)
musinsa_lululemon_reviews['purchase_option_size'] = col.apply(extract_size)

# ---------------------------
# 포맷 정리
# ---------------------------
musinsa_lululemon_reviews['purchase_option_size'] = (
    musinsa_lululemon_reviews['purchase_option_size']
    .fillna(0)
    .astype(int)
    .astype(str)
    .str.zfill(3)
)

In [25]:
musinsa_lululemon_reviews[
    musinsa_lululemon_reviews['purchase_option']
    .astype(str)
    .str.contains(r'\b\d\b', na=False)
]

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,...,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url,women_size_yn,purchase_option_color,purchase_option_size
145,2026-04-21 22:58:17,무신사,None,6142564,스위프틀리 테크 숏슬리브 셔츠 2.0 *웨이스트 렝스,2026-04-17,2026,4,땀자국은 좀 남지만 가볍고 비교적 땀도 잘 마르는것같아서 편해요,4,...,True,lotus lavender · 4,150.0,46.0,None,None,None,0,로터스_라벤더,004
146,2026-04-21 22:58:17,무신사,None,5732869,스위프틀리 테크 숏슬리브 셔츠 2.0 *웨이스트 렝스 SLTE WHT,2026-04-14,2026,4,길이감이 좋아요! 길지않고 색도 너무 맘에들어요!,5,...,True,slate / white · 6,159.0,51.0,None,None,None,0,"슬레이트,화이트",006
147,2026-04-21 22:58:17,무신사,None,5732869,스위프틀리 테크 숏슬리브 셔츠 2.0 *웨이스트 렝스 SLTE WHT,2026-04-12,2026,4,임산부인데요 배부분이 정말 잘 늘어나고 변형도 없어요 재질은 말해모해 짱이에요 다른...,5,...,True,slate / white · 6,168.0,68.0,None,None,None,0,"슬레이트,화이트",006
149,2026-04-21 22:58:17,무신사,None,5732869,스위프틀리 테크 숏슬리브 셔츠 2.0 *웨이스트 렝스 SLTE WHT,2026-03-31,2026,3,운동복으로 최고에요 땀이 금방 마르고 착용감도 좋아요 다른 색도 구매하고 싶어요,5,...,True,slate / white · 4,161.0,50.0,None,None,None,0,"슬레이트,화이트",004
150,2026-04-21 22:58:17,무신사,None,5733162,스위프틀리 테크 숏슬리브 셔츠 2.0 *웨이스트 렝스,2026-03-29,2026,3,필라테스 강사님이나 회원들 대부분 룰루레몬 스위프틀리 소재 통기성이나 신축성 필라랑...,5,...,True,walnut crunch · 4,164.0,49.0,None,None,None,0,월넛_크런치,004
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1928,2026-04-21 23:38:02,무신사,None,5732894,라이트웨이트 하이라이즈 릴랙스드 쇼츠 3inch *롱 라이너,2026-03-25,2026,3,크게 나왔다는데 이너레깅스 올라가는거 싫어서 그냥 평소 사이즈 대로 샀는데 그래도 ...,3,...,False,club blue / white · 6,163.0,52.0,None,None,None,0,"클럽_블루,화이트",006
1929,2026-04-21 23:38:07,무신사,None,5732794,저지 트레이닝 숏슬리브 셔츠 WHT,2026-03-24,2026,3,배송도 빠르고 상품도 너무 좋아요 재구매 의사가 있어요,5,...,False,white · 6,177.0,65.0,None,None,None,0,화이트,006
1930,2026-04-21 23:38:11,무신사,None,5732769,저지 트레이닝 숏슬리브 셔츠 *워드마크 WHT,2026-04-10,2026,4,같은 디자인 으로 다른 색도 샀어요 구김이 안 가서 좋아요,5,...,False,white · 4,160.0,45.0,None,None,None,0,화이트,004
1936,2026-04-21 23:38:40,무신사,None,5732592,러브 탱크 탑 BLK,2026-03-13,2026,3,고민하다가 구매했는데 너무 잘산것 같습니다 만족합니다,5,...,False,black · 8,176.0,80.0,None,None,None,0,블랙,008


### 형 변환

In [27]:
musinsa_lululemon_reviews['has_image'] = musinsa_lululemon_reviews['has_image'].astype(int)


# 형 변환(datetime)
date_cols = ['collect_date']

existing_cols = [col for col in date_cols if col in musinsa_lululemon_reviews.columns]

musinsa_lululemon_reviews[existing_cols] = (
    musinsa_lululemon_reviews[existing_cols]
    .apply(pd.to_datetime, errors='coerce')
)

# 형 변환(int32)
musinsa_lululemon_reviews['helpful_count'] = musinsa_lululemon_reviews['helpful_count'].astype('Int32')

# 형 변환(int16)
musinsa_lululemon_reviews['year'] = musinsa_lululemon_reviews['year'].astype('Int16')

# 형 변환(int8)
int8_cols = ['month', 'rating', 'has_image', 'women_size_yn']

existing_cols = [col for col in int8_cols if col in musinsa_lululemon_reviews.columns]

musinsa_lululemon_reviews[existing_cols] = musinsa_lululemon_reviews[existing_cols].astype('Int8')

# 형 변환(float32)
num_cols = ['user_height', 'user_weight']

existing_cols = [col for col in num_cols if col in musinsa_lululemon_reviews.columns]

musinsa_lululemon_reviews[existing_cols] = (
    musinsa_lululemon_reviews[existing_cols]
    .apply(pd.to_numeric, errors='coerce')
    .astype('float32')
)

# 형 변환(category)
cat_cols = [
    'platform',
    'purchase_option_color',
    'purchase_option_size',
    'user_height_group',
    'user_weight_group'
]

existing_cols = [col for col in cat_cols if col in musinsa_lululemon_reviews.columns]

musinsa_lululemon_reviews[existing_cols] = musinsa_lululemon_reviews[existing_cols].astype('category')

# 형 변환(str)
str_cols = ['review_id', 'product_url', 'product_id']

existing_cols = [col for col in str_cols if col in musinsa_lululemon_reviews.columns]

musinsa_lululemon_reviews[existing_cols] = musinsa_lululemon_reviews[existing_cols].astype(str)

### 컬럼 순서 재정렬

In [28]:
# ======================
# 컬럼 순서 재정렬
# ======================

column_order = [
    'collect_date',
    'platform',
    'review_id',
    'product_id',
    'product_name',
    'review_date',
    'year',
    'month',
    'content',
    'rating',
    'helpful_count',
    'has_image',
    'purchase_option',
    'purchase_option_color',
    'purchase_option_size',
    'women_size_yn',
    'user_height',
    'user_weight',
    'user_height_group',
    'user_weight_group',
    'product_url'
]

# 컬럼 존재하는 것만 선택
musinsa_lululemon_reviews = musinsa_lululemon_reviews[
    [col for col in column_order if col in musinsa_lululemon_reviews.columns]
]

# 확인
musinsa_lululemon_reviews.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,...,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-21 22:57:48,무신사,NaN,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-09,2026,4,룰루레몬 헤어밴드 좋습니다 러닝할때 아주 잘 사용해요,4,...,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN
1,2026-04-21 22:57:48,무신사,NaN,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,땀이 많은 편이라 러닝 한번 하고나면 상의가 반쯤 젖을만큼 땀을 흘리는데 확실히 밴...,5,...,0,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN
2,2026-04-21 22:57:48,무신사,NaN,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,안벗겨지고 깔끔하고 무난합니다 땀도 잘 흡수하고 가볍고 빨리 마르고 착용감도 너어무...,5,...,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN
3,2026-04-21 22:57:48,무신사,NaN,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,벌써 여러 가지 색상으로 룰루레몬에 헤어밴드를 많이 가지고 있습니다 매우 만족하며 ...,5,...,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN
4,2026-04-21 22:57:48,무신사,NaN,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-01,2026,4,운동할 때 쓰려고 샀는데 진짜 너무 좋아요 재질도 너무 만족스럽네요,5,...,0,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN


In [29]:
musinsa_lululemon_reviews.info()

<class 'pandas.DataFrame'>
Index: 1529 entries, 0 to 1943
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   collect_date           1529 non-null   datetime64[us]
 1   platform               1529 non-null   category      
 2   review_id              0 non-null      str           
 3   product_id             1529 non-null   str           
 4   product_name           1529 non-null   str           
 5   review_date            1529 non-null   datetime64[us]
 6   year                   1529 non-null   Int16         
 7   month                  1529 non-null   Int8          
 8   content                1529 non-null   str           
 9   rating                 1529 non-null   Int8          
 10  helpful_count          1529 non-null   Int32         
 11  has_image              1529 non-null   Int8          
 12  purchase_option        1529 non-null   str           
 13  purchase_option_col